In [ ]:
# conda create --name ola --clone qwen
# conda activate ola
# module load cuda/12.2

In [ ]:
import os
os.environ['LOWRES_RESIZE'] = '384x32'
os.environ['HIGHRES_BASE'] = '0x32'
os.environ['VIDEO_RESIZE'] = "0x64"
os.environ['VIDEO_MAXRES'] = "480"
os.environ['VIDEO_MINRES'] = "288"
os.environ['MAXRES'] = '1536'
os.environ['MINRES'] = '0'
os.environ['REGIONAL_POOL'] = '2x'
os.environ['FORCE_NO_DOWNSAMPLE'] = '1'
os.environ['LOAD_VISION_EARLY'] = '1'
os.environ['SKIP_LOAD_VIT'] = '1'
    

import torch
import json
import glob
import csv
import re
from PIL import Image
import numpy as np
import transformers
from tqdm import tqdm
from typing import Dict, Optional, Sequence, List
from ola.conversation import conv_templates, SeparatorStyle, Conversation
from ola.model.builder import load_pretrained_model
from ola.utils import disable_torch_init
from ola.datasets.preprocess import tokenizer_image_token, tokenizer_speech_image_token, tokenizer_speech_question_image_token
from ola.mm_utils import get_model_name_from_path, KeywordsStoppingCriteria, process_anyres_video, process_anyres_highres_image
from ola.constants import IGNORE_INDEX, DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX, DEFAULT_SPEECH_TOKEN

model_path = "THUdyh/Ola-7b"
tokenizer, model, image_processor, _ = load_pretrained_model(model_path, None)
model = model.to('cuda').eval()
model = model.bfloat16()

In [ ]:
text = """For the provided face image, output a JSON object with the following attributes:
        
            {
              "gender": ["male", "female"],
              "age": ["young", "middle-aged", "senior"],
              "skin_color": ["light", "medium", "dark"],
              "ancestry": ["asian", "south_asian", "black", "latino/hispanic", "middle_eastern", "white", "indigenous"],
              "hair_color": ["black", "brown", "red", "blonde", "gray", "other"],
              "bangs": ["yes", "no"],
              "bald": ["yes", "no"],
              "beard": ["no", "mustache", "stubble", "full"],
              "glasses": ["no", "regular", "sun"],
              "headwear": ["no", "beanie", "cap", "hat", "headband", "hijab", "helmet", "turban"],
            }

          Ensure the labeling is based on visible evidence only. If an attribute is unclear, return "unknown".
          
          Only output the JSON without any additional explanation or text.
          
          Example JSON output:
          
          {
            "gender": "female",
            "age": "middle-aged",
            "skin_color": "light",
            "ancestry": "asian",
            "hair_color": "black",
            "bangs": "no",
            "bald": "no",
            "beard": "no",
            "glasses": "sun",
            "headwear": "beanie",
          }
          """

default_keys = ["gender", "age", "skin_color", "ancestry", "hair_color", "bangs", 
                "bald", "beard", "glasses", "headwear"]

# Function to process a single image
def process_images(image_paths):
    # Load images
    images = [Image.open(path) for path in image_paths]
    image_sizes = [img.size for img in images]
    # Prompt setup
    qs = DEFAULT_IMAGE_TOKEN + "\n" + text
    conv = Conversation(
        system="""<|im_start|>system
You are an image analysis tool specialized in facial attribute classification.""",
        roles=("<|im_start|>user", "<|im_start|>tool"),
        version="qwen",
        messages=[],
        offset=0,
        sep_style=SeparatorStyle.CHATML,
        sep="<|im_end|>",
    )
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()
    # Tokenize prompt and repeat it for all images in batch
    input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").to('cuda')
    input_ids = input_ids.unsqueeze(0).repeat(len(images), 1)  # (batch_size, seq_len)
    pad_token_ids = 151643
    attention_masks = input_ids.ne(pad_token_ids).long().to('cuda')
    # Image preprocessing
    image_processor.do_resize = False
    image_processor.do_center_crop = False
    image_tensor, image_highres_tensor = [], []
    for img in images:
        tensor, highres = process_anyres_highres_image(img, image_processor)
        image_tensor.append(tensor)
        image_highres_tensor.append(highres)
    image_tensor = torch.stack(image_tensor).bfloat16().to("cuda")  # (batch_size, C, H, W)
    image_highres_tensor = torch.stack(image_highres_tensor).bfloat16().to("cuda")
    # Dummy speech inputs (repeated for each sample in batch)
    batch_size = len(images)
    speechs = [torch.zeros(1, 3000, 128).bfloat16().to('cuda')] * batch_size
    speech_lengths = [torch.LongTensor([3000]).to('cuda')] * batch_size
    speech_wavs = [torch.zeros([1, 480000]).to('cuda')] * batch_size
    speech_chunks = [torch.LongTensor([1]).to('cuda')] * batch_size
    # Stopping criteria
    stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
    keywords = [stop_str]
    stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)
    # Inference
    with torch.inference_mode():
        output_ids = model.generate(
            inputs=input_ids,
            images=image_tensor,
            images_highres=image_highres_tensor,
            image_sizes=image_sizes,
            modalities=['image'],
            speech=speechs,
            speech_lengths=speech_lengths,
            speech_chunks=speech_chunks,
            speech_wav=speech_wavs,
            attention_mask=attention_masks,
            use_cache=True,
            stopping_criteria=[stopping_criteria],
            do_sample=False,
            num_beams=1,
            max_new_tokens=1024,
        )
    # Decode outputs
    outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    result = []
    for i in range(len(image_paths)):
        # Extracting the JSON part
        output = outputs[i]
        output = output.strip()
        if output.endswith(stop_str):
            output = output[:-len(stop_str)]
        json_output = re.search(r"\{.*\}", output.strip(), re.DOTALL).group()
        json_output = re.sub(r',\s*}', '}', json_output)  # This removes a comma just before a closing curly brace
        data = json.loads(json_output)
        result.append([image_paths[i].replace('../../../data/RFW/data/', '')] + [data.get(key, "unknown") for key in default_keys])
    torch.cuda.empty_cache()
    return result

In [ ]:
init = 0
files = glob.glob('../../../data/RFW/data/*/*/*')

import os
os.makedirs("../../../results/annotations", exist_ok=True)
output_file = "../../../results/annotations/ola_labels.csv"

# Column names
columns = ["File Name", "Gender", "Age", "Skin Color", "Ancestry", "Hair Color", 
           "Bangs", "Bald", "Beard", "Glasses", "Headwear"]

# Chunk settings
chunk_size = 1  # Number of images processed per batch

# # Write CSV header
with open(output_file, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(columns)  # CSV Header

# Process in chunks
for start_idx in tqdm(range(init, len(files), chunk_size), desc="Processing Images"):
    end_idx = min(start_idx + chunk_size, len(files))
    chunk = files[start_idx:end_idx]
    results = process_images(chunk)
    # Save chunk results to CSV
    with open(output_file, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(results)